# Definições

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
import numpy as np
import os
from tqdm import tqdm

from dataloader import AntiUAVRGBTDataset, collate_fn_superior
from SuperiorDETR import SuperiorDETR

# ============================================================
# CONFIGURAÇÕES
# ============================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
NUM_EPOCHS = 300
ROOT_DIR = r"C:\Users\Micro\Documents\sourcecode\Anti-UAV-RGBT"
LEARNING_RATE = 1e-4

# No seu script de treino, ajuste assim:
transform = {
    "visible": transforms.Compose([
        transforms.ToTensor(), # Converte para 0-1
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Escala para a ResNet
    ]),
    "infrared": transforms.Compose([
        transforms.ToTensor(), # Converte para 0-1
        transforms.Normalize([0.449], [0.226]) # Escala para o canal térmico
    ])
}

# ============================================================
# UTILITÁRIOS DE COORDENADAS (Respeitando seu esboço)
# ============================================================

def box_cxcywh_to_xyxy(x):
    cx, cy, w, h = x.unbind(-1)
    return torch.stack([cx - 0.5 * w, cy - 0.5 * h, cx + 0.5 * w, cy + 0.5 * h], dim=-1)

def box_xywh_to_xyxy(x):
    # Dataloader retorna [x, y, w, h] (top-left)
    x_tl, y_tl, w, h = x.unbind(-1)
    return torch.stack([x_tl, y_tl, x_tl + w, y_tl + h], dim=-1)

def calculate_iou(boxes1, boxes2):
    """ boxes1: [N, 4] xyxy, boxes2: [N, 4] xyxy """
    lt = torch.max(boxes1[:, :2], boxes2[:, :2])
    rb = torch.min(boxes1[:, 2:], boxes2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, 0] * wh[:, 1]
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1 + area2 - inter + 1e-6
    return inter / union

# ============================================================
# CRITÉRIO COM MATCHER DINÂMICO (UNIFICADO)
# ============================================================

class MultimodalCriterion(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, outputs, batch, exist_weight=1.0):
        # 1. Recuperar predições do modelo
        pred_boxes = outputs["pred_boxes"]  
        pred_exist = outputs["exist"]       
        orig_sizes_vis, orig_sizes_ir = outputs["orig_sizes"] 

        # 2. Recuperar Ground Truths (GT)
        gt_vis = batch["boxes_vis"].to(DEVICE)
        gt_ir = batch["boxes_ir"].to(DEVICE)
        exist_vis = batch["exist_vis"].to(DEVICE)
        exist_ir = batch["exist_ir"].to(DEVICE)

        B, T, N, _ = pred_boxes.shape
        preds_all_xyxy = box_cxcywh_to_xyxy(pred_boxes)

        metrics = {
            "loss": torch.tensor(0.0, device=DEVICE),
            "iou_all": 0.0, "msa_all": 0.0,
            "iou_vis": [], "iou_ir": [], "count": 0
        }

        for b in range(B):
            w_v, h_v = float(orig_sizes_vis[b][0]), float(orig_sizes_vis[b][1])
            w_i, h_i = float(orig_sizes_ir[b][0]), float(orig_sizes_ir[b][1])
            
            scale_v = torch.tensor([w_v, h_v, w_v, h_v], dtype=torch.float32, device=DEVICE)
            scale_i = torch.tensor([w_i, h_i, w_i, h_i], dtype=torch.float32, device=DEVICE)
            
            gt_v_norm = box_xywh_to_xyxy(gt_vis[b]) / scale_v
            gt_i_norm = box_xywh_to_xyxy(gt_ir[b]) / scale_i
            
            target = torch.zeros((T, 4), device=DEVICE)
            mask_v = exist_vis[b] > 0
            mask_i = exist_ir[b] > 0
            
            target[mask_i] = gt_i_norm[mask_i]
            target[mask_v] = gt_v_norm[mask_v]
            
            valid_mask = mask_v | mask_i 

            if valid_mask.any():
                v_preds = preds_all_xyxy[b][valid_mask]
                v_target = target[valid_mask].unsqueeze(1)
                
                # MATCHER
                dist = torch.abs(v_preds - v_target).sum(-1)
                best_indices = dist.argmin(dim=-1)
                
                best_preds = v_preds[torch.arange(v_preds.size(0)), best_indices]
                
                # Perda de Caixa (L1)
                loss_box = F.l1_loss(best_preds, target[valid_mask])
                
                # Perda de Existência
                target_exist = torch.zeros((v_preds.size(0), N), device=DEVICE)
                target_exist[torch.arange(v_preds.size(0)), best_indices] = 1.0
                v_exist = pred_exist[b][valid_mask]
                
                loss_ce = F.binary_cross_entropy(v_exist.clamp(1e-7, 1.0 - 1e-7), target_exist)
                
                # --- AJUSTE 1: Penalização de Background ---
                # Força as queries não selecionadas a ficarem próximas de zero
                bg_mask = target_exist == 0
                loss_bg = (v_exist[bg_mask] ** 2).mean()
                
                # Combinação da perda de existência com o peso de warm-up
                loss_exist = exist_weight * (loss_ce + 0.1 * loss_bg)
                
                metrics["loss"] += (loss_box + loss_exist)
                
                # MÉTRICAS
                iou = calculate_iou(best_preds, target[valid_mask])
                metrics["iou_all"] += iou.mean().item()
                metrics["msa_all"] += (iou > 0.5).float().mean().item()
                metrics["count"] += 1

                if mask_v.any():
                    idx_v_in_valid = torch.where(mask_v[valid_mask])[0]
                    if len(idx_v_in_valid) > 0:
                        iou_v = calculate_iou(best_preds[idx_v_in_valid], gt_v_norm[mask_v])
                        metrics["iou_vis"].append(iou_v.mean().item())

                if mask_i.any():
                    idx_i_in_valid = torch.where(mask_i[valid_mask])[0]
                    if len(idx_i_in_valid) > 0:
                        iou_i = calculate_iou(best_preds[idx_i_in_valid], gt_i_norm[mask_i])
                        metrics["iou_ir"].append(iou_i.mean().item())

        denom = metrics["count"] + 1e-6
        return {
            "loss": metrics["loss"] / denom,
            "iou_global": metrics["iou_all"] / denom,
            "msa_global": metrics["msa_all"] / denom,
            "iou_vis": np.mean(metrics["iou_vis"]) if metrics["iou_vis"] else 0.0,
            "iou_ir": np.mean(metrics["iou_ir"]) if metrics["iou_ir"] else 0.0
        }

# ============================================================
# FUNÇÃO DE EXECUÇÃO DE ÉPOCA
# ============================================================

def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE, exist_weight=1.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    
    epoch_logs = {
        "loss": [], "iou_global": [], "msa_global": [], 
        "iou_vis": [], "iou_ir": [], 
        "gate_vis_avg": [], "gate_ir_avg": []
    }

    desc = ">> TREINO" if is_train else ">> VALIDAÇÃO"
    pbar = tqdm(loader, desc=desc)
    
    for batch in pbar:
        vis, ir = batch["vis_frames"], batch["ir_frames"]
        
        with torch.set_grad_enabled(is_train):
            outputs = model(vis, ir)
            # Passamos o exist_weight para o critério aqui
            res = criterion(outputs, batch, exist_weight=exist_weight)

            if is_train:
                optimizer.zero_grad()
                res["loss"].backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
                optimizer.step()

        for k in res.keys():
            if k in epoch_logs:
                val = res[k].item() if torch.is_tensor(res[k]) else res[k]
                epoch_logs[k].append(val)
        
        for g_key in ["gate_vis_avg", "gate_ir_avg"]:
            if g_key in outputs:
                val = outputs[g_key].item() if torch.is_tensor(outputs[g_key]) else outputs[g_key]
                epoch_logs[g_key].append(val)
        
        pbar.set_postfix({
            "Loss": f"{np.mean(epoch_logs['loss']):.4f}",
            "IoU": f"{np.mean(epoch_logs['iou_global']):.4f}",
            "G_V": f"{np.mean(epoch_logs['gate_vis_avg']):.2f}" if epoch_logs['gate_vis_avg'] else "N/A"
        })

    return {k: np.mean(v) if len(v) > 0 else 0.0 for k, v in epoch_logs.items()}

# ============================================================
# SCRIPT DE EXECUÇÃO PRINCIPAL
# ============================================================
if __name__ == "__main__":
    train_dataset = AntiUAVRGBTDataset(ROOT_DIR, split="train", transform=transform)
    val_dataset = AntiUAVRGBTDataset(ROOT_DIR, split="test", transform=transform)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
        collate_fn=collate_fn_superior, num_workers=0
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
        collate_fn=collate_fn_superior, num_workers=0
    )

    model = SuperiorDETR(d_model=256, n_queries=20).to(DEVICE)
    criterion = MultimodalCriterion()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    os.makedirs("checkpoints", exist_ok=True)
    RESUME_PATH = "checkpoints/superior_detr_checkpoint.pth"
    start_epoch = 0
    best_msa = 0.0

    if os.path.exists(RESUME_PATH):
        print(f"\n[*] Encontrado checkpoint: {RESUME_PATH}. Carregando...")
        checkpoint = torch.load(RESUME_PATH, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_msa = checkpoint['best_msa']
        print(f"[+] Retomando da Época {start_epoch+1}")

    for epoch in range(start_epoch, NUM_EPOCHS):
        print(f"\n{'='*30} ÉPOCA {epoch+1}/{NUM_EPOCHS} {'='*30}")

        # --- AJUSTE 2: Lógica de Warm-up de Existência ---
        # Nas primeiras 10 épocas, focamos 80% na caixa e 20% na existência.
        # Após a época 10, usamos o peso total (1.0).
        current_exist_weight = 0.2 if epoch < 10 else 1.0
        if epoch == 10: print("\n[!] Warm-up finalizado. Peso de existência agora é 1.0")

        # Executa Treino com o peso atual
        train_results = run_epoch(model, train_loader, criterion, 
                                  optimizer=optimizer, exist_weight=current_exist_weight)

        # Executa Validação (sempre peso 1.0 para métricas reais)
        val_results = run_epoch(model, val_loader, criterion, 
                                optimizer=None, exist_weight=1.0)

        percent_padding_train = 100 * train_dataset.padding_count / train_dataset.total_calls
        percent_padding_val = 100 * val_dataset.padding_count / val_dataset.total_calls

        print(f"{percent_padding_train:.2f}% padding no treino.")
        print(f"{percent_padding_val:.2f}% padding na validação.")
        print("-" * 40)
        print(f"\n[RESULTADOS DA ÉPOCA {epoch+1}]")
        print(f"{'MÉTRICA':<12} | {'TREINO':<10} | {'VALIDAÇÃO':<10}")
        print("-" * 40)
        
        for k in train_results.keys():
            print(f"{k.upper():<12} | {train_results[k]:.4f}      | {val_results[k]:.4f}")

        current_checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_msa': best_msa,
        }
        torch.save(current_checkpoint, RESUME_PATH)

        if val_results['msa_global'] > best_msa:
            best_msa = val_results['msa_global']
            torch.save(model.state_dict(), "checkpoints/superior_detr_best.pth")
            print(f"\n[!] Novo recorde de MSA: {best_msa:.4f}. Peso salvo.")


============================== ÉPOCA 1/300 ==============================


>> VALIDAÇÃO: 100%|██████████| 12/12 [00:52<00:00,  4.34s/it, Loss=0.1505, IoU=0.0226, G_V=0.90]



[RESULTADOS DA ÉPOCA 1]
MÉTRICA      | TREINO     | VALIDAÇÃO 
0.62% padding no treino.
0.00% padding na validação.
----------------------------------------
LOSS         | 0.1358      | 0.1505
IOU_GLOBAL   | 0.0259      | 0.0226
MSA_GLOBAL   | 0.0069      | 0.0042
IOU_VIS      | 0.0259      | 0.0226
IOU_IR       | 0.0188      | 0.0044
GATE_VIS_AVG | 0.9000      | 0.9000
GATE_IR_AVG  | 0.9000      | 0.9000

[!] Novo recorde de MSA: 0.0042. Peso salvo.

============================== ÉPOCA 2/300 ==============================


>> VALIDAÇÃO: 100%|██████████| 12/12 [00:51<00:00,  4.26s/it, Loss=0.1513, IoU=0.0782, G_V=0.90]



[RESULTADOS DA ÉPOCA 2]
MÉTRICA      | TREINO     | VALIDAÇÃO 
0.62% padding no treino.
0.00% padding na validação.
----------------------------------------
LOSS         | 0.0912      | 0.1513
IOU_GLOBAL   | 0.0289      | 0.0782
MSA_GLOBAL   | 0.0006      | 0.0208
IOU_VIS      | 0.0289      | 0.0785
IOU_IR       | 0.0155      | 0.0119
GATE_VIS_AVG | 0.9000      | 0.9000
GATE_IR_AVG  | 0.9000      | 0.9000

[!] Novo recorde de MSA: 0.0208. Peso salvo.

============================== ÉPOCA 3/300 ==============================


...

>> VALIDAÇÃO: 100%|██████████| 12/12 [00:51<00:00,  4.26s/it, Loss=0.0571, IoU=0.3764, G_V=0.90]



[RESULTADOS DA ÉPOCA 10]
MÉTRICA      | TREINO     | VALIDAÇÃO 
0.62% padding no treino.
0.00% padding na validação.
----------------------------------------
LOSS         | 0.0202      | 0.0571
IOU_GLOBAL   | 0.4387      | 0.3764
MSA_GLOBAL   | 0.4044      | 0.3392
IOU_VIS      | 0.4391      | 0.3876
IOU_IR       | 0.0372      | 0.0492
GATE_VIS_AVG | 0.9000      | 0.9000
GATE_IR_AVG  | 0.9000      | 0.9000

============================== ÉPOCA 11/300 ==============================

[!] Warm-up finalizado. Peso de existência agora é 1.0


>> VALIDAÇÃO: 100%|██████████| 12/12 [00:50<00:00,  4.20s/it, Loss=0.0525, IoU=0.4376, G_V=0.90]



[RESULTADOS DA ÉPOCA 11]
MÉTRICA      | TREINO     | VALIDAÇÃO 
0.62% padding no treino.
0.00% padding na validação.
----------------------------------------
LOSS         | 0.0313      | 0.0525
IOU_GLOBAL   | 0.4455      | 0.4376
MSA_GLOBAL   | 0.4275      | 0.4597
IOU_VIS      | 0.4458      | 0.4374
IOU_IR       | 0.0249      | 0.0242
GATE_VIS_AVG | 0.9000      | 0.9000
GATE_IR_AVG  | 0.9000      | 0.9000

[!] Novo recorde de MSA: 0.4597. Peso salvo.

============================== ÉPOCA 12/300 ==============================


>> TREINO:  60%|██████    | 12/20 [01:59<01:22, 10.28s/it, Loss=0.0298, IoU=0.4876, G_V=0.90]

# LOG

# 